# 03.6 COCO Polygon → YOLO Instance Segmentation Labels

**목표**
1. `scripts/06A_convert_coco_to_yolo_seg.py` 사용 (서버에 없으면 이 노트북으로 생성)
2. COCO JSON의 `segmentation` polygon을 추출
3. YOLO-seg 포맷(`.txt`)으로 변환 (`category_id=26` → class `0`)

**중요:** 노트북에서 `%run` / `main()` / `parse_args()` 를 쓰지 마세요.
Jupyter가 `-f kernel.json`을 넘겨 argparse가 깨집니다. `run_conversion()`만 호출하세요.

**주의**
- 기존 detect용 `data/yolo/*/labels`는 건드리지 않음
- 출력은 `data/yolo_seg/` 아래로 분리

## 0) 경로 확인

In [ ]:
from pathlib import Path
import json

PROJECT_DIR = Path.cwd().parent
DATA_DIR = PROJECT_DIR / "data"
SCRIPTS_DIR = PROJECT_DIR / "scripts"

TRAIN_JSON = DATA_DIR / "arcade/arcade/stenosis/train/annotations/train.json"
VAL_JSON = DATA_DIR / "arcade/arcade/stenosis/val/annotations/val.json"
OUT_ROOT = DATA_DIR / "yolo_seg"
SCRIPT_PATH = SCRIPTS_DIR / "06A_convert_coco_to_yolo_seg.py"

PIPELINES = ["original", "clahe", "clahe_gamma"]
TARGET_COCO_ID = 26
YOLO_CLASS_ID = 0

print("PROJECT_DIR :", PROJECT_DIR, "exists=", PROJECT_DIR.exists())
print("TRAIN_JSON  :", TRAIN_JSON, "exists=", TRAIN_JSON.exists())
print("VAL_JSON    :", VAL_JSON, "exists=", VAL_JSON.exists())
print("SCRIPTS_DIR :", SCRIPTS_DIR, "exists=", SCRIPTS_DIR.exists())
print("OUT_ROOT    :", OUT_ROOT)

## 1) COCO JSON에서 segmentation 구조 확인 (먼저 빼보기)

ARCADE stenosis annotation은 보통 아래 형태입니다.

```python
{
  'category_id': 26,
  'segmentation': [[x1, y1, x2, y2, ...]],  # polygon (픽셀 좌표)
  'bbox': [x, y, w, h],
  'iscrowd': 0
}
```

In [ ]:
with open(TRAIN_JSON, "r", encoding="utf-8") as f:
    coco = json.load(f)

print("keys:", coco.keys())
print("images:", len(coco["images"]))
print("annotations:", len(coco["annotations"]))
print("categories (last 3):", coco["categories"][-3:])

# stenosis only
stenosis_anns = [a for a in coco["annotations"] if a.get("category_id") == TARGET_COCO_ID]
print("stenosis anns:", len(stenosis_anns))

ann0 = stenosis_anns[0]
print("\n=== sample annotation ===")
print("keys:", ann0.keys())
print("category_id:", ann0["category_id"])
print("iscrowd:", ann0.get("iscrowd"))
print("bbox:", ann0["bbox"])
print("segmentation type:", type(ann0["segmentation"]).__name__)
print("n_polygons:", len(ann0["segmentation"]))
print("first polygon n_coords:", len(ann0["segmentation"][0]))
print("first 12 coords:", ann0["segmentation"][0][:12])

# 타입 점검: polygon list vs RLE dict
seg_types = {}
for a in stenosis_anns:
    seg = a.get("segmentation")
    seg_types[type(seg).__name__] = seg_types.get(type(seg).__name__, 0) + 1
print("\nsegmentation type counts:", seg_types)
print("iscrowd==1 count:", sum(1 for a in stenosis_anns if a.get("iscrowd", 0) == 1))

## 2) 서버에 변환 스크립트 파일 생성

로컬 Cursor에만 있던 `.py`를 서버 `ARCADE/scripts/`에 직접 씁니다.

In [ ]:
SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)

SCRIPT_SOURCE = r'''#!/usr/bin/env python3
"""COCO polygon segmentation -> YOLO-seg labels (ARCADE stenosis)."""

from __future__ import annotations

import argparse
import json
import shutil
from collections import defaultdict
from pathlib import Path
from typing import Any

TARGET_COCO_ID = 26
YOLO_CLASS_ID = 0
MIN_POLYGON_POINTS = 3


def clip01(v: float) -> float:
    return min(max(v, 0.0), 1.0)


def normalize_polygon(flat_xy, img_w: int, img_h: int):
    if img_w <= 0 or img_h <= 0:
        return None
    if len(flat_xy) < MIN_POLYGON_POINTS * 2 or len(flat_xy) % 2 != 0:
        return None

    out = []
    for i in range(0, len(flat_xy), 2):
        x = clip01(float(flat_xy[i]) / img_w)
        y = clip01(float(flat_xy[i + 1]) / img_h)
        out.extend([x, y])

    xs, ys = out[0::2], out[1::2]
    if max(xs) - min(xs) <= 0.0 or max(ys) - min(ys) <= 0.0:
        return None
    return out


def polygon_to_yolo_line(norm_xy, class_id: int = YOLO_CLASS_ID) -> str:
    coords = " ".join(f"{v:.6f}" for v in norm_xy)
    return f"{class_id} {coords}"


def extract_polygons(segmentation: Any):
    if segmentation is None or isinstance(segmentation, dict):
        return []
    if not isinstance(segmentation, list) or len(segmentation) == 0:
        return []
    if isinstance(segmentation[0], (int, float)):
        return [list(map(float, segmentation))]

    polys = []
    for poly in segmentation:
        if isinstance(poly, list) and poly and isinstance(poly[0], (int, float)):
            polys.append(list(map(float, poly)))
    return polys


def convert_coco_to_yolo_seg_labels(
    json_path: Path,
    target_coco_id: int = TARGET_COCO_ID,
    yolo_class_id: int = YOLO_CLASS_ID,
):
    with open(json_path, "r", encoding="utf-8") as f:
        coco = json.load(f)

    img_dict = {
        img["id"]: {
            "file_name": img["file_name"],
            "w": int(img["width"]),
            "h": int(img["height"]),
        }
        for img in coco["images"]
    }
    yolo_labels = {img["file_name"]: [] for img in coco["images"]}
    stats = defaultdict(int)

    for ann in coco["annotations"]:
        if ann.get("category_id") != target_coco_id:
            stats["skipped_wrong_category"] += 1
            continue
        if ann.get("iscrowd", 0) == 1:
            stats["skipped_iscrowd"] += 1
            continue

        image_id = ann["image_id"]
        if image_id not in img_dict:
            stats["skipped_missing_image"] += 1
            continue

        info = img_dict[image_id]
        file_name = info["file_name"]
        img_w, img_h = info["w"], info["h"]

        polys = extract_polygons(ann.get("segmentation"))
        if not polys:
            stats["skipped_empty_or_rle"] += 1
            continue

        wrote_any = False
        for poly in polys:
            norm = normalize_polygon(poly, img_w, img_h)
            if norm is None:
                stats["skipped_invalid_polygon"] += 1
                continue
            yolo_labels[file_name].append(polygon_to_yolo_line(norm, yolo_class_id))
            stats["polygons_written"] += 1
            wrote_any = True

        if wrote_any:
            stats["instances_kept"] += 1

    n_empty = sum(1 for v in yolo_labels.values() if len(v) == 0)
    print(f"[{json_path.name}]")
    print(f"  images              : {len(yolo_labels)}")
    print(f"  stenosis instances  : {stats['instances_kept']}")
    print(f"  polygons written    : {stats['polygons_written']}")
    print(f"  empty label files   : {n_empty}")
    for k in (
        "skipped_wrong_category",
        "skipped_iscrowd",
        "skipped_empty_or_rle",
        "skipped_invalid_polygon",
        "skipped_missing_image",
    ):
        if stats[k]:
            print(f"  {k}: {stats[k]}")
    return yolo_labels


def write_labels(yolo_labels, label_dir: Path, dry_run: bool = False) -> int:
    label_dir.mkdir(parents=True, exist_ok=True)
    n = 0
    for file_name, lines in yolo_labels.items():
        txt_path = label_dir / (Path(file_name).stem + ".txt")
        if dry_run:
            n += 1
            continue
        with open(txt_path, "w", encoding="utf-8") as f:
            f.write("\n".join(lines))
            if lines:
                f.write("\n")
        n += 1
    return n


def mirror_labels_to_pipelines(src_split_dir, out_root, pipelines, split_name, dry_run=False):
    for pipe in pipelines:
        dst = out_root / pipe / "labels" / split_name
        if dry_run:
            print(f"  [dry-run] would copy -> {dst}")
            continue
        dst.mkdir(parents=True, exist_ok=True)
        for txt in src_split_dir.glob("*.txt"):
            shutil.copy2(txt, dst / txt.name)
        print(f"  copied labels -> {dst} ({len(list(dst.glob('*.txt')))} files)")
'''

if not SCRIPT_PATH.exists():
    SCRIPT_PATH.write_text(SCRIPT_SOURCE, encoding="utf-8")
else:
    print("Existing converter retained:", SCRIPT_PATH)
print("Wrote:", SCRIPT_PATH)
print("Size :", SCRIPT_PATH.stat().st_size, "bytes")
print("Exists:", SCRIPT_PATH.exists())

## 3) import 확인 + dry-run 변환 (파일 저장 없음)

기대치(기존 detect 변환과 동일 규모):
- train: images=1000, stenosis≈1625, empty≈3
- val: images=200, stenosis≈406, empty≈0

In [ ]:
import importlib.util

# 파일명이 06A_로 시작해 일반 import가 어려움 -> 경로로 직접 로드
# 절대 %run / main() / parse_args() 쓰지 말 것 (Jupyter -f 충돌)
spec = importlib.util.spec_from_file_location("coco_to_yolo_seg", SCRIPT_PATH)
conv = importlib.util.module_from_spec(spec)
spec.loader.exec_module(conv)

print("module file:", SCRIPT_PATH, "exists=", SCRIPT_PATH.exists())

# dry-run: 통계만 (파일 저장 없음)
train_labels = conv.convert_coco_to_yolo_seg_labels(TRAIN_JSON)
val_labels = conv.convert_coco_to_yolo_seg_labels(VAL_JSON)

# 샘플 YOLO-seg 라인 확인 (detect와 다르게 좌표가 6개 이상)
sample_name, sample_lines = next(
    (k, v) for k, v in train_labels.items() if len(v) > 0
)
print("\n=== sample YOLO-seg label ===")
print("image:", sample_name)
print("n_instances:", len(sample_lines))
parts = sample_lines[0].split()
print("class_id:", parts[0])
print("n_values_after_class:", len(parts) - 1, "(should be even, >= 6)")
print("line preview:", sample_lines[0][:120], "...")

## 4) 실제 라벨 파일 저장 (`data/yolo_seg/`)

dry-run 숫자가 기대치와 맞으면 아래 셀만 실행하세요.

`DRY_RUN = False`로 저장됩니다.

In [ ]:
DRY_RUN = False  # True면 저장 안 함

staging = OUT_ROOT / "_labels_shared"

for split_name, labels in (("train", train_labels), ("val", val_labels)):
    split_dir = staging / split_name
    n = conv.write_labels(labels, split_dir, dry_run=DRY_RUN)
    print(f"[{split_name}] wrote/planned={n} -> {split_dir}")
    conv.mirror_labels_to_pipelines(
        split_dir, OUT_ROOT, PIPELINES, split_name, dry_run=DRY_RUN
    )

print("\nDone. Example paths:")
for pipe in PIPELINES:
    p = OUT_ROOT / pipe / "labels" / "train"
    print(f"  {p} exists={p.exists()} n_txt={len(list(p.glob('*.txt'))) if p.exists() else 0}")

## 5) 저장 결과 스팟체크

- detect 라벨: `0 cx cy w h` (숫자 5개)
- seg 라벨: `0 x1 y1 x2 y2 ...` (숫자 6개 이상, 짝수)

In [ ]:
sample_txts = sorted((OUT_ROOT / "original" / "labels" / "train").glob("*.txt"))
print("n_train_labels:", len(sample_txts))

shown = 0
for txt in sample_txts:
    text = txt.read_text(encoding="utf-8").strip()
    if not text:
        continue
    line = text.splitlines()[0]
    n_nums = len(line.split()) - 1
    print(f"{txt.name}: values_after_class={n_nums}")
    print(" ", line[:140], "...")
    shown += 1
    if shown >= 3:
        break

empty = [t for t in sample_txts if t.read_text(encoding="utf-8").strip() == ""]
print("empty label files (background):", len(empty))